# Book Database Compiler

In [1]:
import re
import csv
import json
import sqlite3
import numpy as np
import pandas as pd

from IPython.display import clear_output

#### Parse author data.

The `author` dataset has the least amount of data so it'll be easier to filter the remaining two datasets using this one.

In [2]:
def format_id(text: str) -> str:
    return text.split('/')[-1]


def extract_name(text: str):
    try:
        info_dict = json.loads(text)
        return info_dict.get('name', np.nan)
    except Exception as e:
        return np.nan

In [3]:
conn = sqlite3.connect('./datasets/books.db')
conn.execute('DROP TABLE IF EXISTS authors')

chunk_size = 100_000
file_path = './datasets/authors.txt'

for i, chunk in enumerate(pd.read_csv(file_path, sep="\t", chunksize=chunk_size, quoting=csv.QUOTE_NONE, header=None)):
    df = chunk
    df.columns = ['type', 'id', 'revision', 'last_modified', 'info']
    df['author_name'] = df['info'].apply(extract_name)
    df['id'] = df['id'].apply(format_id)
    df = df.dropna(subset=['author_name'])
    df = df[['id', 'author_name']]
    
    df.to_sql('authors', conn, if_exists='append', index=False)
    print(f"chunk {(i + 1)}: kept {len(df)} of {chunk_size}")

conn.close()

#### Parse works data.

Cross reference author id's in `works` with author ids in `authors`.

In [4]:
def get_title(text: str):
    try:
        info_dict = json.loads(text)
        return info_dict.get('title', np.nan)
    except Exception:
        return np.nan


def get_author_ids(text: str, key_id: str):
    try:
        info_dict = json.loads(text)
        return [format_id(k['author']['key']) for k in info_dict.get(key_id, [])]
    except Exception:
        return []


def get_subjects(text: str):
    try:
        info_dict = json.loads(text)
        return info_dict.get('subjects', [])
    except Exception:
        return []


def get_covers(text: str):
    try:
        info_dict = json.loads(text)
        return info_dict.get('covers', [])
    except Exception:
        return []

In [5]:
conn = sqlite3.connect('./datasets/books.db')
conn.execute('DROP TABLE IF EXISTS works')

chunk_size = 100_000
file_path = './datasets/works.txt'

author_ids = set(pd.read_sql('SELECT id FROM authors', conn)['id'])

for i, chunk in enumerate(pd.read_csv(file_path, sep="\t", chunksize=chunk_size, quoting=csv.QUOTE_NONE, header=None)):
    df = chunk
    df.columns = ['type', 'id', 'revision', 'last_modified', 'info']
    
    df['subjects'] = df['info'].apply(get_subjects)
    df['cover'] = df['info'].apply(get_covers)

    df = df[df['subjects'].apply(len) > 0]
    df = df[df['cover'].apply(len) > 0]

    df['subjects'] = df['subjects'].apply(lambda s: ",".join(s))
    df['cover'] = df['cover'].apply(lambda c: c[0])
    
    df['title'] = df['info'].apply(get_title)
    df['id'] = df['id'].apply(format_id)
    
    df['work_author_ids'] = df['info'].apply(get_author_ids, key_id='authors')
    df = df[df['work_author_ids'].apply(lambda ids: any(a for a in ids if a in author_ids))]
    df['work_author_ids'] = df['work_author_ids'].apply(lambda ids: ",".join(ids))
    
    df = df.dropna(subset=['title'])
    df = df[['id', 'info', 'title', 'subjects', 'cover', 'work_author_ids']]
    
    df.to_sql('works', conn, if_exists='append', index=False)
    
    clear_output(wait=True)
    print(f"chunk {(i + 1)}: kept {len(df)} of {chunk_size}")

conn.close()

#### Parse editions data.

Cross reference work ids in `editions` with work ids in `works`.

In [7]:
def get_work_ids(text: str, key_id: str) -> list:
    try:
        info_dict = json.loads(text)
        return [format_id(k['key']) for k in info_dict.get(key_id, [])]
    except Exception as e:
        return []


def get_isbn(text: str):
    try:
        info_dict = json.loads(text)
        isbn_13 = info_dict.get('isbn_13')
        if isbn_13:
            return isbn_13[0]
        isbn_10 = info_dict.get('isbn_10')
        if isbn_10:
            return isbn_10[0]
        return np.nan
    except Exception:
        return np.nan


def is_english(text: str):
    try:
        info_dict = json.loads(text)
        for k in info_dict.get('languages', []):
            if format_id(k['key']) == 'eng':
                return True
        
        return False
    except Exception as e:
        return False

In [8]:
conn = sqlite3.connect('./datasets/books.db')
conn.execute('DROP TABLE IF EXISTS editions')

chunk_size = 100_000
file_path = './datasets/editions.txt'

work_ids = set(pd.read_sql('SELECT id FROM works', conn)['id'])

for i, chunk in enumerate(pd.read_csv(file_path, sep="\t", chunksize=chunk_size, quoting=csv.QUOTE_NONE, header=None)):
    df = chunk
    df.columns = ['type', 'id', 'revision', 'last_modified', 'info']
    df = df[df['info'].apply(is_english)]
    df['id'] = df['id'].apply(format_id)
    df['isbn'] = df['info'].apply(get_isbn)
    df['title'] = df['info'].apply(get_title)
    df['edition_work_ids'] = df['info'].apply(get_work_ids, key_id='works')
    df = df[df['edition_work_ids'].apply(lambda ids: any(w for w in ids if w in work_ids))]
    df['edition_work_ids'] = df['edition_work_ids'].apply(lambda ids: ",".join(ids))
    df = df.dropna(subset=['title'])
    df = df[['id', 'title', 'edition_work_ids', 'isbn']]
    
    df.to_sql('editions', conn, if_exists='append', index=False)
    
    clear_output(wait=True)
    print(f"chunk {(i + 1)}: kept {len(df)} of {chunk_size}")

conn.close()

chunk 568: kept 7246 of 100000
